# Imports & paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from lifetimes import BetaGeoFitter

DATA_INTERIM = Path("../data/interim")
DATA_PROCESSED = Path("../data/processed")
ART_MODELS = Path("../artifacts/models")
ART_SCORES = Path("../artifacts/scores")

# Load data

In [2]:
df_txn = pd.read_parquet(DATA_INTERIM / "transactions_clean.parquet")

# Tính x, t_x, T cho BG-NBD

In [3]:
from lifetimes.utils import summary_data_from_transaction_data

summary = summary_data_from_transaction_data(
    df_txn,
    customer_id_col="customer_id",
    datetime_col="transaction_date",
    monetary_value_col="amount",
    observation_period_end="2025-12-31"
)
summary = summary.reset_index() 

In [4]:
from lifetimes import BetaGeoFitter

bgf = BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(
    summary["frequency"],
    summary["recency"],
    summary["T"]
)

summary["p_alive"] = bgf.conditional_probability_alive(
    summary["frequency"],
    summary["recency"],
    summary["T"]
)

In [5]:
summary["exp_txn_90d"] = bgf.conditional_expected_number_of_purchases_up_to_time(
    90,
    summary["frequency"],
    summary["recency"],
    summary["T"]
)

# Huấn luyện mô hình Gamma–Gamma

In [6]:
from lifetimes import GammaGammaFitter

ggf = GammaGammaFitter(penalizer_coef=0.01)

summary_gg = summary[summary["monetary_value"] > 0]

ggf.fit(
    summary_gg["frequency"],
    summary_gg["monetary_value"]
)

summary.loc[summary_gg.index, "expected_avg_order_value"] = (
    ggf.conditional_expected_average_profit(
        summary_gg["frequency"],
        summary_gg["monetary_value"]
    )
)

# Tính CLV cho từng khách hàng

CLV = expected purchases × expected order value × margin

In [7]:
summary["CLV_3m"] = ggf.customer_lifetime_value(
    bgf,
    summary["frequency"],
    summary["recency"],
    summary["T"],
    summary["monetary_value"],
    time=3,              
    discount_rate=0.01,   
    freq="D"
)


In [8]:
summary.head()

,customer_id,frequency,recency,T,monetary_value,p_alive,exp_txn_90d,expected_avg_order_value,CLV_3m
0,C00000,11.0,112.0,112.0,93.370000,0.970418,7.934405,94.412757,734.718249
1,C00001,17.0,278.0,289.0,68.373529,0.958491,5.037573,68.926945,340.473208
2,C00002,9.0,37.0,133.0,86.307778,0.000266,0.001586,87.520289,0.136095
3,C00003,3.0,45.0,88.0,19.393333,0.538345,1.962628,21.243203,40.891244
4,C00004,17.0,98.0,206.0,109.965882,0.000132,0.000945,110.718219,0.102606


In [9]:
summary.sort_values("CLV_3m", ascending=False).head(10)[
    ["customer_id", "p_alive", "exp_txn_90d", "expected_avg_order_value", "CLV_3m"]
]

,customer_id,p_alive,exp_txn_90d,expected_avg_order_value,CLV_3m
851,C00884,0.985551,46.017459,338.612608,15291.701034
1809,C01876,0.987189,49.572162,305.620024,14867.378503
285,C00295,0.984018,42.861114,288.186670,12122.094103
659,C00688,0.969874,48.113036,244.436664,11541.942219
2829,C02934,0.980457,46.678717,229.423878,10507.246524
234,C00243,0.985074,41.198406,212.951238,8609.264030
2254,C02338,0.986419,45.687926,179.405439,8043.494499
2389,C02478,0.987542,45.316437,161.935754,7200.735283
151,C00158,0.995257,47.076813,153.036748,7065.842154
2810,C02914,0.992110,47.507477,134.857415,6284.970481


# Log

In [10]:

bgf.save_model(ART_MODELS / "bgnbd_model.pkl")
ggf.save_model(ART_MODELS / "gamma_gamma_model.pkl")

out = summary[[
    "customer_id", 
    "p_alive", 
    "exp_txn_90d", 
    "expected_avg_order_value", 
    "CLV_3m"
]].copy()

out.to_parquet(
    ART_SCORES / "bgnbd_gamma_gamma_90d.parquet",
    index=False
)